# ogrenci_04 — Kaggle CPU deneyleri

**Run All**, yalnız seçili ve onaylı aşamadaki bekleyen koşuları oturum ve grup süre bütçeleri elverdiğince sırayla çalıştırır. Sonraki aşamaya geçmez. Ön kontrol ve tüm senaryo tablosu hesap hücresinden önce üretilir; Run All hesap hücresini de çalıştırır. OpenMC, Kaggle kernelinden ayrı ve sürümü doğrulanan bir Python ortamında çalışır.


In [ ]:
NUCLEAR_DATASET_ROOT = "/kaggle/input/NUKLEER_VERI_DATASET_ADI"
CODE_DATASET_ROOT = "/kaggle/input/datasets/ascanipek/openmc-code-base"
SELECTED_STAGE = "stage_32"  # stage_32, stage_128 veya stage_256
SESSION_BUDGET_SECONDS = 4 * 3600  # kullanıcı oturum tavanı; Kaggle platform garantisi değildir
import time
SESSION_STARTED_MONOTONIC = time.monotonic()


## 1. Ortam ve kod kontrolü
Kaggle kernelindeki `mamba` Conda paket yöneticisi değildir. Hash'i sabitlenmiş resmî micromamba ile OpenMC 0.16.0 ayrı bir geçici ortamda kurulur. Internet açık olmalıdır. Ortam ve paket bütünlüğü doğrulanmadan hesap başlamaz; kurulum ortamı /kaggle/working çıktısına eklenmez.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import urllib.request

code_root = Path(CODE_DATASET_ROOT).resolve()
if not code_root.is_dir():
    raise FileNotFoundError(f"Kod Dataset'i bulunamadı: {code_root}")
ledger_path = code_root / "bundle_hashes.json"
adapter = code_root / "kaggle_runner.py"
if not ledger_path.is_file() or not adapter.is_file():
    raise FileNotFoundError(f"Kod paketi eksik: {code_root}")
ledger = json.loads(ledger_path.read_text(encoding="utf-8"))
if hashlib.sha256(adapter.read_bytes()).hexdigest() != ledger["files"]["kaggle_runner.py"]:
    raise ValueError("Kod adaptörü hash uyuşmazlığı")
if "kaggle_runner" in sys.modules:
    loaded_path = Path(sys.modules["kaggle_runner"].__file__).resolve()
    if loaded_path != adapter:
        raise RuntimeError(f"Farklı konumdan kod yüklenmiş: {loaded_path}. Kernel'i yeniden başlat.")
sys.path.insert(0, str(code_root))
import kaggle_runner as kr
print("Doğrulanan paket dosyası:", kr.verify_bundle(code_root))

micromamba = Path("/tmp/kaggle_micromamba")
micromamba_url = ("https://github.com/mamba-org/micromamba-releases/"
                  "releases/download/2.9.0-0/micromamba-linux-64")
micromamba_sha256 = "366cd9cd8be14df1ab8ed50352a82111082a36686b2d389fdb79a92c3fafb3e3"
if not micromamba.is_file():
    urllib.request.urlretrieve(micromamba_url, micromamba)
if hashlib.sha256(micromamba.read_bytes()).hexdigest() != micromamba_sha256:
    raise ValueError("micromamba indirme hash'i uyuşmuyor")
micromamba.chmod(0o755)
openmc_prefix = Path("/tmp/openmc_016")
openmc_python = openmc_prefix / "bin/python"
child_env = os.environ.copy()
child_env["MAMBA_ROOT_PREFIX"] = "/tmp/kaggle_micromamba_root"
if not (openmc_prefix / "bin/openmc").is_file():
    subprocess.run([str(micromamba), "create", "-y", "-p", str(openmc_prefix),
                    "-c", "conda-forge", "python=3.13", "openmc=0.16.0",
                    "numpy", "h5py", "pandas", "matplotlib"],
                   env=child_env, check=True)
if not openmc_python.is_file():
    raise FileNotFoundError(f"OpenMC Python ortamı bulunamadı: {openmc_python}")

def openmc_process(source, *args):
    process_env = child_env.copy()
    process_env["PYTHONPATH"] = str(code_root)
    return subprocess.run([str(openmc_python), "-u", "-c", source,
                           *(str(arg) for arg in args)],
                          env=process_env, check=True)

openmc_process("import kaggle_runner as kr; print('OpenMC ortamı:', kr.ensure_openmc())")


## 2. Seçili aşama ve ön kontrol
Kod/paket hashleri, teknik onay, merkezi atama, geometri, gerekli izotoplar, termal saçılma ve sıcaklık kapsamı ayrı OpenMC ortamında doğrulanır. Bu hücre taşıma hesabı başlatmaz; bütün atanmış senaryoları gösterir.


In [ ]:
import pandas as pd

preflight_code = """
import sys
from pathlib import Path
import kaggle_runner as kr

code_root, data_root, stage = Path(sys.argv[1]), Path(sys.argv[2]), sys.argv[3]
if stage not in kr.STAGES:
    raise ValueError(f"Geçersiz aşama: {stage}")
kr.verify_bundle(code_root)
kr.ensure_openmc()
project = code_root / "project"
students, runner, _, _ = kr._modules(project)
package = kr._package(project, stage)
info = students.verify_package(package, require_approval=True)
manifest = kr._json(package / "manifest.json")
assignment = kr._json(project / "configs/v2_student_assignment_v1.json")
registry = assignment["packages"].get(info["package_id"])
if (info.get("owner") != kr.OWNER or info.get("stage") != stage or
        registry is None or registry.get("owner") != kr.OWNER or
        registry.get("stage") != stage or
        registry.get("scenario_ids") != [case["scenario_id"] for case in manifest["experiments"]] or
        registry.get("package_sha256") != kr._sha(package / "package.json") or
        registry.get("manifest_sha256") != kr._sha(package / "manifest.json")):
    raise ValueError("Merkezi atama veya Kaggle paketi uyuşmuyor")
expected = {"stage_32": 8, "stage_128": 24, "stage_256": 32}[stage]
if len(manifest["experiments"]) != expected:
    raise ValueError("Atanmış senaryo sayısı uyuşmuyor")
index = kr.cross_sections_index(data_root, Path("/kaggle/working"))
checks = runner.preflight(package)
if len(checks["cases"]) != expected or checks["transport_started"]:
    raise ValueError("Ön kontrol eksik veya transport işaretlenmiş")
print("Paket:", package)
print("Nükleer veri indeksi:", index["working_index"])
print("Ön kontrol geçen koşu:", len(checks["cases"]))
print("Transport başladı mı:", checks["transport_started"])
"""
openmc_process(preflight_code, code_root, NUCLEAR_DATASET_ROOT, SELECTED_STAGE)

package = (code_root / "project/student_packages/v2_students_portable_v1"
           / SELECTED_STAGE / "ogrenci_04"
           / f"v2_{SELECTED_STAGE}_ogrenci_04_beavrs_v2_pilot_scope_v1")
manifest = json.loads((package / "manifest.json").read_text(encoding="utf-8"))
rows = [{"experiment_id": case["experiment_id"],
         "scenario_id": case["scenario_id"],
         **case["controls"], **case["calculation"]}
        for case in manifest["experiments"]]
print("Seçili aşama:", SELECTED_STAGE)
print("Atanmış koşu:", len(rows))
print("Onaylı grup süre tavanı (s):", manifest["group_runtime_budget_seconds"])
print("Bu oturumun süre tavanı (s):", SESSION_BUDGET_SECONDS)
display(pd.DataFrame(rows))


## 3. Seçili aşamadaki eksik koşuları süre bütçeleri içinde çalıştır
Bu hücre OpenMC hesabını **sırayla ve bekleyerek** yürütür; arka plan işçisi oluşturmaz. Önceki Output Input olarak eklenmişse tamamlanan geçerli koşular doğrulanır. Sıradaki koşu kalan süreye sığmayacak görünüyorsa yeni hesap başlatmadan teslim ZIP'ini hazırlar. Save & Run All temiz bir Kaggle oturumunda baştan yürür; önceki Output eklenmeden eski etkileşimli oturumu devralmaz.


In [ ]:
run_code = """
import sys
from pathlib import Path
import kaggle_runner as kr

code_root, data_root, stage = Path(sys.argv[1]), Path(sys.argv[2]), sys.argv[3]
session_budget, session_started = float(sys.argv[4]), float(sys.argv[5])
context = kr.prepare(data_root, code_root, stage)
print("Grup:", context["group"], flush=True)
print("Doğrulanmış tamamlanan:", len(context["completed"]), flush=True)
print("Bekleyen:", len(context["manifest"]["experiments"]) - len(context["completed"]), flush=True)
result = kr.run_stage(context, session_budget, session_started)
print("Durum:", result["state"], flush=True)
print("Tamamlanan:", result["completed"], "Bekleyen:", result["pending"], flush=True)
print("Durma nedeni:", result.get("stop_reason"), flush=True)
print("Tam çıktılar:", result.get("full_outputs"), flush=True)
print("Teslim ZIP:", result.get("delivery_zip"), flush=True)
"""
openmc_process(run_code, code_root, NUCLEAR_DATASET_ROOT, SELECTED_STAGE,
               SESSION_BUDGET_SECONDS, SESSION_STARTED_MONOTONIC)
